# **1. Import Libraries**

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import gradio as gr
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# **2. Load MNIST Dataset**

In [2]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


# **3. Normalize data & Reshape for CNN**

In [3]:
x_train = x_train / 255.0
x_test = x_test / 255.0

x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# **4. Build CNN Model**

In [4]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(10, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# **5. Compile Model**

In [5]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


# **6. Train the Model**

In [6]:
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(x_train)

model.fit(
    datagen.flow(x_train, y_train, batch_size=64),
    epochs=10,
    validation_data=(x_test, y_test)
)

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 37s 31ms/step - accuracy: 0.8685 - loss: 0.4282 - val_accuracy: 0.9783 - val_loss: 0.0741
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.9607 - loss: 0.1283 - val_accuracy: 0.9883 - val_loss: 0.0373
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 22s 23ms/step - accuracy: 0.9693 - loss: 0.1007 - val_accuracy: 0.9908 - val_loss: 0.0301
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 22s 23ms/step - accuracy: 0.9755 - loss: 0.0805 - val_accuracy: 0.9925 - val_loss: 0.0238
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.9794 - loss: 0.0697 - val_accuracy: 0.9939 - val_loss: 0.0181
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.9804 - loss: 0.0653 - val_accuracy: 0.9895 - val_loss: 0.0294
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 22s 23ms/step - accuracy: 0.9820 - loss: 0.0612 - val_accuracy: 0.9911 - val_loss: 0.0262
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.9821 - loss: 0.0588 - 

# **7. Predict the Digit**

In [7]:

def predict_digit(img):
    try:
        if isinstance(img, dict):
            img = img["composite"]

        img = Image.fromarray(img)
        img = img.convert("L")
        img = img.resize((28, 28))
        img = np.array(img)
        img = 255 - img
        img = img / 255.0
        img = img.reshape(1, 28, 28, 1)

        prediction = model.predict(img)[0]

        return {str(i): float(prediction[i]) for i in range(10)}

    except Exception as e:
        return {"error": str(e)}

# **8. Gradio UI to interact with the model**

In [8]:
interface = gr.Interface(
    fn=predict_digit,
    inputs=gr.Sketchpad(),
    outputs=gr.Label(num_top_classes=3),
    title="Handwritten Digit Recognizer",
    description="Draw a number (0–9) and the model will predict it!"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f1537c9be9ee123af.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
